In [1]:

import pandas as pd

# 1. Dosya Yolu
dosya_yolu = '/kaggle/input/datasets/mansiaggarwal88/ai-engineering-github-repositories/ai_engineering_ecosystem_intelligence.csv'

# 2. Veriyi Belleğe Alma
df = pd.read_csv(dosya_yolu)

# 3. Temel Özet
print("--- 1. Veri Seti Boyutu ---")
print(f"Toplam Satır: {df.shape[0]} | Toplam Kolon: {df.shape[1]}\n")

print("--- 2. Veri Tipleri ve Bellek Kullanımı ---")
df.info()

print("\n--- 3. İlk 5 Satır Önizlemesi ---")
display(df.head())

print("\n--- 4. Eksik Veri (Null) Analizi ---")
missing_data = df.isnull().sum()
missing_summary = missing_data[missing_data > 0].sort_values(ascending=False)

if not missing_summary.empty:
    print(missing_summary)
else:
    print("Veri setinde hiçbir eksik (null) değer bulunmuyor.")

--- 1. Veri Seti Boyutu ---
Toplam Satır: 30198 | Toplam Kolon: 37

--- 2. Veri Tipleri ve Bellek Kullanımı ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30198 entries, 0 to 30197
Data columns (total 37 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   full_name                  30198 non-null  object 
 1   owner                      30198 non-null  object 
 2   repo_name                  30198 non-null  object 
 3   description                29795 non-null  object 
 4   html_url                   30198 non-null  object 
 5   language                   28352 non-null  object 
 6   stars                      30198 non-null  int64  
 7   forks                      30198 non-null  int64  
 8   open_issues                30198 non-null  int64  
 9   size_kb                    30198 non-null  int64  
 10  license                    21903 non-null  object 
 11  license_family             30198 non-null  obj

,full_name,owner,repo_name,description,html_url,language,stars,forks,open_issues,size_kb,...,has_readme_file,has_contributing_file,has_code_of_conduct,has_issue_template,has_license_file,releases_count,latest_release_date,release_cadence_days,dependency_manifest_found,detected_dependencies
0,microsoft/markitdown,microsoft,markitdown,Python tool for converting files and office do...,https://github.com/microsoft/markitdown,Python,171310,12462,840,4331,...,True,False,True,False,True,20.0,2026-07-29T18:19:15Z,30.9,none,NaN
1,langchain-ai/langchain,langchain-ai,langchain,The agent engineering platform.,https://github.com/langchain-ai/langchain,Python,143380,23882,448,586424,...,True,True,True,False,True,100.0,2026-07-30T14:56:10Z,1.0,none,NaN
2,bytedance/deer-flow,bytedance,deer-flow,An open-source long-horizon SuperAgent harness...,https://github.com/bytedance/deer-flow,Python,79210,10815,950,54195,...,True,True,True,False,True,1.0,2026-06-25T16:23:22Z,NaN,none,NaN
3,headroomlabs-ai/headroom,headroomlabs-ai,headroom,"Compress tool outputs, logs, files, and RAG ch...",https://github.com/headroomlabs-ai/headroom,Python,64467,4902,618,70481,...,True,True,True,False,True,100.0,2026-07-29T23:10:22Z,0.9,pyproject.toml,langchain;crewai;autogen;openai;anthropic;tran...
4,BerriAI/litellm,BerriAI,litellm,"The fastest, litest AI Gateway. Rust core with...",https://github.com/BerriAI/litellm,Python,55484,10302,4722,1304291,...,True,True,False,False,True,100.0,2026-08-03T19:18:02Z,0.8,package.json,NaN



--- 4. Eksik Veri (Null) Analizi ---
detected_dependencies        29016
release_cadence_days         27842
latest_release_date          27617
has_code_of_conduct          26198
has_contributing_file        26198
has_readme_file              26198
github_health_pct            26198
releases_count               26198
has_license_file             26198
has_issue_template           26198
dependency_manifest_found    26198
framework_stack              21679
license                       8295
language                      1846
description                    403
dtype: int64


In [2]:
# 1. Veri setindeki tüm kolon isimlerini listeleme
print("--- Veri Setindeki Tüm Kolonlar ---")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

--- Veri Setindeki Tüm Kolonlar ---
1. full_name
2. owner
3. repo_name
4. description
5. html_url
6. language
7. stars
8. forks
9. open_issues
10. size_kb
11. license
12. license_family
13. topics
14. search_topic_matched
15. created_at
16. updated_at
17. pushed_at
18. archived
19. visibility
20. repo_age_days
21. days_since_last_push
22. maintenance_status
23. popularity_tier
24. ai_category
25. framework_stack
26. deep_enrichment_applied
27. github_health_pct
28. has_readme_file
29. has_contributing_file
30. has_code_of_conduct
31. has_issue_template
32. has_license_file
33. releases_count
34. latest_release_date
35. release_cadence_days
36. dependency_manifest_found
37. detected_dependencies


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

data = df.copy()
data['description_length'] = data['description'].fillna('').apply(len)
data['topics_count'] = data['topics'].fillna('').apply(lambda x: len(str(x).split(',')) if x else 0)
data['dependencies_count'] = data['detected_dependencies'].fillna('').apply(lambda x: len(str(x).split(',')) if x else 0)

# KESİN ÇÖZÜM: Karmaşık (str/bool) kolonları tamamen sayısal 0 ve 1'e çeviriyoruz
boolean_features = ['archived', 'visibility', 'has_readme_file', 'has_contributing_file', 'has_code_of_conduct', 'has_issue_template', 'has_license_file', 'dependency_manifest_found']
for col in boolean_features:
    if col in data.columns:
        # Tüm kolonu metne çevir, küçük harf yap. Sadece 'true' olanlara 1, gerisine 0 bas.
        data[col] = data[col].astype(str).str.lower()
        data[col] = data[col].apply(lambda x: 1 if x == 'true' else 0)

cols_to_drop = [
    'full_name', 'owner', 'repo_name', 'html_url', 'forks', 'open_issues',
    'popularity_tier', 'github_health_pct', 'created_at', 'updated_at',
    'pushed_at', 'latest_release_date', 'search_topic_matched',
    'deep_enrichment_applied', 'license_family', 'description', 'topics', 'detected_dependencies'
]
data = data.drop(columns=[col for col in cols_to_drop if col in data.columns], errors='ignore')

X = data.drop(columns=['stars'])
y = data['stars']
y_log = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)
print("Hücre 1 Tamam: Mantıksal veriler başarıyla 0 ve 1'e dönüştürüldü.")

Hücre 1 Tamam: Mantıksal veriler başarıyla 0 ve 1'e dönüştürüldü.


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

ordinal_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# Mantıksal kolonlar (0/1) artık sayısal (num) gruba dahil edildi
numeric_features = [
    'size_kb', 'repo_age_days', 'days_since_last_push', 'releases_count', 
    'release_cadence_days', 'description_length', 'topics_count', 'dependencies_count',
    'archived', 'visibility', 'has_readme_file', 'has_contributing_file', 
    'has_code_of_conduct', 'has_issue_template', 'has_license_file', 'dependency_manifest_found'
]
numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])

categorical_mode_features = ['language']
categorical_mode_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('ordinal', ordinal_enc)])

categorical_constant_features = ['framework_stack', 'license', 'ai_category', 'maintenance_status']
categorical_constant_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')), ('ordinal', ordinal_enc)])

# Boolean transformere gerek kalmadı, yapı çok daha sade
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat_mode', categorical_mode_transformer, categorical_mode_features),
        ('cat_const', categorical_constant_transformer, categorical_constant_features)
    ], remainder='drop'
)
print("Hücre 2 Tamam: Preprocessor kuralları güncellendi ve sadeleştirildi.")

Hücre 2 Tamam: Preprocessor kuralları güncellendi ve sadeleştirildi.


In [5]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Preprocessor ve XGBoost'u tek boruya bağla
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1))
])

print("Model eğitiliyor...")
model_pipeline.fit(X_train, y_train)

# Tahmin yap ve Logaritmayı geri çevir (Gerçek Star sayısını görmek için)
y_pred_log = model_pipeline.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

# Metrikleri Hesapla
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred))
r2 = r2_score(y_test_real, y_pred)

print(f"\nR2 Skoru: {r2:.4f}")
print(f"RMSE: {rmse:.2f}")

Model eğitiliyor...

R2 Skoru: 0.0858
RMSE: 4624.05
